In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import pandas as pd 
import config 
import era5_clean as ec
import feature_engineering as fe

In [2]:
ds=ec.load_raw_era5()
ds

Opened 1 of 72 file(s)
Dimensions: {'valid_time': 744, 'latitude': 231, 'longitude': 176}
Variables : ['t2m', 'tp', 'swvl1', 'u10', 'v10']


<xarray.Dataset> Size: 605MB
Dimensions:     (valid_time: 744, latitude: 231, longitude: 176)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 6kB 2020-01-01 ... 2020-01-31T23:...
    expver      (valid_time) <U4 12kB ...
  * latitude    (latitude) float64 2kB 28.5 28.4 28.3 28.2 ... 5.8 5.7 5.6 5.5
  * longitude   (longitude) float64 1kB 92.0 92.1 92.2 ... 109.3 109.4 109.5
    number      int64 8B ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 121MB ...
    tp          (valid_time, latitude, longitude) float32 121MB ...
    swvl1       (valid_time, latitude, longitude) float32 121MB ...
    u10         (valid_time, latitude, longitude) float32 121MB ...
    v10         (valid_time, latitude, longitude) float32 121MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-10-11T07:26 GRIB to CDM+CF via cfgrib-0.9.1...

In [3]:
hourly = ec.to_dataframe(ds)
print(f"{len(hourly):,} hourly records")
hourly.head()

19,123,776 hourly records


,time,latitude,longitude,t2m,tp,swvl1,u10,v10,grid_id
0,2020-01-01,28.5,92.0,254.704315,0.000010,0.324631,0.881989,0.318176,28.5_92.0
1,2020-01-01,28.5,92.1,256.055878,0.000005,0.323914,1.054840,-0.049988,28.5_92.1
2,2020-01-01,28.5,92.2,256.684784,0.000004,0.306915,1.049957,-0.357605,28.5_92.2
3,2020-01-01,28.5,92.3,257.294159,0.000007,0.307159,0.755035,-0.563660,28.5_92.3
4,2020-01-01,28.5,92.4,257.977753,0.000014,0.293030,0.344879,-0.726746,28.5_92.4


In [4]:
print(f"300 K      -> {ec.kelvin_to_celsius(300):.2f} °C   (expected: 26.85)")
print(f"0.024 m    -> {ec.metres_to_mm(0.024):.1f} mm   (expected: 24.0)")
print(f"10 m/s     -> {ec.ms_to_kmh(10):.1f} km/h (expected: 36.0)")

300 K      -> 26.85 °C   (expected: 26.85)
0.024 m    -> 24.0 mm   (expected: 24.0)
10 m/s     -> 36.0 km/h (expected: 36.0)


In [5]:
wind_cmp=ec.compare_wind_methods(hourly)
wind_cmp.head(10)

,grid_id,date,speed_correct_ms,speed_wrong_ms,difference_ms,understated_by_pct
0,10.0_105.1,2020-01-01,2.817357,2.439335,0.378,13.400000
1,10.0_105.1,2020-01-02,2.951587,2.545645,0.406,13.800000
2,10.0_105.1,2020-01-03,2.296183,1.986199,0.310,13.500000
3,10.0_105.1,2020-01-04,1.971725,1.409629,0.562,28.500000
4,10.0_105.1,2020-01-05,2.082542,1.606343,0.476,22.900000
5,10.0_105.1,2020-01-06,1.881010,1.060225,0.821,43.599998
6,10.0_105.1,2020-01-07,2.438354,2.314149,0.124,5.100000
7,10.0_105.1,2020-01-08,2.699418,2.395098,0.304,11.300000
8,10.0_105.1,2020-01-09,1.470607,0.714337,0.756,51.400002
9,10.0_105.1,2020-01-10,1.660262,0.602133,1.058,63.700001


In [6]:
print(f"Average difference: {wind_cmp['understated_by_pct'].mean():.1f}%")
print(f"Maximum difference: {wind_cmp['understated_by_pct'].max():.1f}%")

worst = wind_cmp.loc[wind_cmp["understated_by_pct"].idxmax()]

print()
print("Day with the largest difference:")
print(f"  Correct: {worst.speed_correct_ms:.2f} m/s")
print(f"  Incorrect: {worst.speed_wrong_ms:.2f} m/s")

Average difference: 25.2%
Maximum difference: 99.9%

Day with the largest difference:
  Correct: 0.94 m/s
  Incorrect: 0.00 m/s


In [7]:
daily=ec.aggregate_daily(hourly)
daily.head()

Hourly rows 19,123,776 -> daily rows 796,824
Columns: ['grid_id', 'date', 'soil_moisture', 'latitude', 'longitude', 'temp_max_c', 'temp_mean_c', 'rainfall_mm', 'wind_speed_kmh', 'year', 'month']


,grid_id,date,soil_moisture,latitude,longitude,temp_max_c,temp_mean_c,rainfall_mm,wind_speed_kmh,year,month
0,10.0_105.1,2020-01-01,0.220519,10.0,105.1,31.776031,27.286469,0.977355,10.142485,2020,1
1,10.0_105.1,2020-01-02,0.217468,10.0,105.1,31.545807,27.415649,0.002140,10.625712,2020,1
2,10.0_105.1,2020-01-03,0.215768,10.0,105.1,31.021637,27.089996,0.176883,8.266259,2020,1
3,10.0_105.1,2020-01-04,0.214773,10.0,105.1,30.405670,26.685822,0.198914,7.098211,2020,1
4,10.0_105.1,2020-01-05,0.214426,10.0,105.1,30.203033,25.988983,0.214418,7.497152,2020,1


In [8]:
rain_cmp=ec.compare_rainfall_methods(hourly)
rain_cmp[rain_cmp['max_within_day_mm'] > 0].head(10)

,grid_id,date,sum_all_hours_mm,max_within_day_mm,value_at_23h_mm,sum_overstates_by_x
0,10.0_105.1,2020-01-01,0.984240,0.977355,0.002146,458.700012
1,10.0_105.1,2020-01-02,0.016719,0.002140,0.000882,19.000000
2,10.0_105.1,2020-01-03,3.482728,0.176883,0.176883,19.700001
3,10.0_105.1,2020-01-04,3.756315,0.198914,0.198901,18.900000
4,10.0_105.1,2020-01-05,4.334143,0.214418,0.214418,20.200001
5,10.0_105.1,2020-01-06,0.226381,0.214418,0.000852,265.600006
6,10.0_105.1,2020-01-07,2.711634,0.168622,0.168622,16.100000
7,10.0_105.1,2020-01-08,4.649185,0.243098,0.243098,19.100000
8,10.0_105.1,2020-01-09,6.650756,0.432754,0.432754,15.400000
9,10.0_105.1,2020-01-10,7.626036,1.495734,1.495734,5.100000


In [9]:
daily = fe.add_rolling_features(daily)

one_cell = daily[daily.grid_id == daily.grid_id.iloc[0]]
one_cell[["date","rainfall_mm","rainfall_7d_mm"]].head(10)

Added rainfall_7d_mm
  154,224 row(s) have less than 7 days of history


,date,rainfall_mm,rainfall_7d_mm
0,2020-01-01,0.977355,0.977355
1,2020-01-02,0.002140,0.979495
2,2020-01-03,0.176883,1.156378
3,2020-01-04,0.198914,1.355292
4,2020-01-05,0.214418,1.569710
5,2020-01-06,0.214418,1.784128
6,2020-01-07,0.168622,1.952749
7,2020-01-08,0.243098,1.218492
8,2020-01-09,0.432754,1.649105
9,2020-01-10,1.495734,2.967957


In [10]:
wrong=daily.sort_values("date").copy()
wrong["rainfall_7d_WRONG"] = wrong["rainfall_mm"].rolling(7, min_periods=1).sum()

check = wrong.sort_values(["grid_id","date"]).groupby("grid_id").head(3)
check[["grid_id","date","rainfall_mm","rainfall_7d_mm","rainfall_7d_WRONG"]]

,grid_id,date,rainfall_mm,rainfall_7d_mm,rainfall_7d_WRONG
0,10.0_105.1,2020-01-01,0.977355,0.977355,0.977355
1,10.0_105.1,2020-01-02,0.002140,0.979495,24.413714
2,10.0_105.1,2020-01-03,0.176883,1.156378,47.412031
31,10.0_105.2,2020-01-01,0.975135,0.975135,6.939191
32,10.0_105.2,2020-01-02,0.002140,0.977275,8.087003
...,...,...,...,...,...
796763,9.9_99.0,2020-01-02,2.046669,4.093328,8.547300
796764,9.9_99.0,2020-01-03,0.918752,5.012080,10.380399
796793,9.9_99.1,2020-01-01,2.402726,2.402726,11.929277
796794,9.9_99.1,2020-01-02,2.118522,4.521248,16.238409


In [11]:
daily = fe.add_dry_days(daily)

one_cell = daily[daily.grid_id == daily.grid_id.iloc[0]]
one_cell[["date","rainfall_mm","dry_days"]].head(12)

Added dry_days (threshold 1.0 mm)
  longest dry spell: 31 days


,date,rainfall_mm,dry_days
0,2020-01-01,0.977355,1
1,2020-01-02,0.002140,2
2,2020-01-03,0.176883,3
3,2020-01-04,0.198914,4
4,2020-01-05,0.214418,5
5,2020-01-06,0.214418,6
6,2020-01-07,0.168622,7
7,2020-01-08,0.243098,8
8,2020-01-09,0.432754,9
9,2020-01-10,1.495734,0


In [12]:
daily_full, report = ec.clean_era5_folder()
daily_full = fe.add_weather_features(daily_full)
daily_full.head()


  [1/72] era5_southeast_asia_202001.nc... -> 796,824 daily rows
  [2/72] era5_southeast_asia_202002.nc... -> 745,416 daily rows
  [3/72] era5_southeast_asia_202003.nc... -> 796,824 daily rows
  [4/72] era5_southeast_asia_202004.nc... -> 771,120 daily rows
  [5/72] era5_southeast_asia_202005.nc... -> 796,824 daily rows
  [6/72] era5_southeast_asia_202006.nc... -> 771,120 daily rows
  [7/72] era5_southeast_asia_202007.nc... -> 796,824 daily rows
  [8/72] era5_southeast_asia_202008.nc... -> 796,824 daily rows
  [9/72] era5_southeast_asia_202009.nc... -> 771,120 daily rows
  [10/72] era5_southeast_asia_202010.nc... -> 796,824 daily rows
  [11/72] era5_southeast_asia_202011.nc... -> 771,120 daily rows
  [12/72] era5_southeast_asia_202012.nc... -> 796,824 daily rows
  [13/72] era5_southeast_asia_202101.nc... -> 796,824 daily rows
  [14/72] era5_southeast_asia_202102.nc... -> 719,712 daily rows
  [15/72] era5_southeast_asia_202103.nc... -> 796,824 daily rows
  [16/72] era5_southeast_asia_2021

,grid_id,date,soil_moisture,latitude,longitude,temp_max_c,temp_mean_c,rainfall_mm,wind_speed_kmh,year,month,rainfall_7d_mm,dry_days
0,10.0_105.1,2020-01-01,0.220519,10.0,105.1,31.776031,27.286469,0.977355,10.142485,2020,1,0.977355,1
1,10.0_105.1,2020-01-02,0.217468,10.0,105.1,31.545807,27.415649,0.002140,10.625712,2020,1,0.979495,2
2,10.0_105.1,2020-01-03,0.215768,10.0,105.1,31.021637,27.089996,0.176883,8.266259,2020,1,1.156378,3
3,10.0_105.1,2020-01-04,0.214773,10.0,105.1,30.405670,26.685822,0.198914,7.098211,2020,1,1.355292,4
4,10.0_105.1,2020-01-05,0.214426,10.0,105.1,30.203033,25.988983,0.214418,7.497152,2020,1,1.569710,5


In [13]:
import schema

present = [c for c in daily_full.columns if c in schema.SCHEMA]
missing = [c for c in schema.SCHEMA if c not in daily_full.columns]

print(f"Columns matching the schema: {len(present)}")
print(f"Still missing              : {missing}")
print()
print("(The missing columns will be added in Sessions 09–11)")

Columns matching the schema: 13
Still missing              : ['country', 'fire', 'hotspot_count', 'frp_max', 'frp_sum', 'land_cover_dominant', 'forest_pct', 'cropland_pct', 'grassland_pct', 'other_pct']

(The missing columns will be added in Sessions 09–11)


In [14]:
path=ec.save_daily(daily_full)
reloaded=pd.read_parquet(path)
print(f"Reloaded: {len(reloaded):,} rows")

Saved 56,343,168 rows -> D:\Duanduhoche.Huy\wildfire-risk-analysis\data\processed\weather_daily.parquet
Reloaded: 56,343,168 rows
